## simple Gen AI app Using Langchain

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GOOGLE_API_KEY']=os.getenv('GOOGLE_API_KEY')
## langsmith tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [3]:
## Data Ingestion
from langchain_community.document_loaders import WebBaseLoader


In [4]:
loader=WebBaseLoader("https://medium.com/@musabzafar63/building-large-ai-systems-is-an-architecture-problem-not-a-model-problem-42ab622e712f")
loader

In [5]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://medium.com/@musabzafar63/building-large-ai-systems-is-an-architecture-problem-not-a-model-problem-42ab622e712f', 'title': 'Medium', 'description': 'Building Large AI Systems Is an Architecture Problem Not a Model Problem Why your clean 50-line GenAI prototype turned into a 3,000 line god class and the five architectural pillars required to fix …', 'language': 'en'}, page_content="MediumBuilding Large AI Systems Is an Architecture Problem Not a Model Problem | by Musab Zafar | Oct, 2026 | MediumSitemapOpen in appSign upSign inMedium LogoGet appWriteSearchSign upSign inWhy your clean 50-line GenAI prototype turned into a 3,000 line god class and the five architectural pillars required to fix it.Domain-Driven Design Establishing Bounded ContextsData-Driven Programming Decoupling Policy from CodeClean Architecture: Protecting Domain Logic from Provider ChurnApplying Inversion of Control to LLM GatewaysEvent-Driven Architecture Decoupling Synchronous L

In [6]:
### Load Data---> Docs-->Divide text into chunks--->vectors--->vector embeddings---->vector store db
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
documents=text_splitter.split_documents(docs)

In [7]:
documents

[Document(metadata={'source': 'https://medium.com/@musabzafar63/building-large-ai-systems-is-an-architecture-problem-not-a-model-problem-42ab622e712f', 'title': 'Medium', 'description': 'Building Large AI Systems Is an Architecture Problem Not a Model Problem Why your clean 50-line GenAI prototype turned into a 3,000 line god class and the five architectural pillars required to fix …', 'language': 'en'}, page_content='MediumBuilding Large AI Systems Is an Architecture Problem Not a Model Problem | by Musab Zafar | Oct, 2026 | MediumSitemapOpen in appSign upSign inMedium LogoGet appWriteSearchSign upSign inWhy your clean 50-line GenAI prototype turned into a 3,000 line god class and the five architectural pillars required to fix it.Domain-Driven Design Establishing Bounded ContextsData-Driven Programming Decoupling Policy from CodeClean Architecture: Protecting Domain Logic from Provider ChurnApplying'),
 Document(metadata={'source': 'https://medium.com/@musabzafar63/building-large-ai-s

In [13]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [14]:
from langchain_community.vectorstores import FAISS
vectorstoredb = FAISS.from_documents(documents, embeddings)

In [15]:
vectorstoredb

In [17]:
query="The problem isn’t the model"
result=vectorstoredb.similarity_search(query)
result[0].page_content

'transactions, HTTP retry loops, and analytics tracking are hopelessly tangled together.The problem isn’t the model. The problem is treating an AI product as a simple API wrapper rather than a distributed software system.State-of-the-art results in production are rarely achieved by swapping in a marginally larger model. They are achieved through system design: orchestrating multiple interacting components, retrievers, specialized evaluators, and deterministic guardrails.To build an AI codebase'

In [39]:
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv()
# Use an active, supported model version
llm=ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite", 
    temperature=0.7, 
    timeout=15
)

In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """Answer the following question based only on the provided context:
<context>
{context}
</context>

Question: {input}"""
)


document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)


result = document_chain.invoke({
    "input": "What is the major trap of moving logic to configuration?",
    "context": [
        Document(
            page_content="Moving logic to configuration carries a major trap: never embed multi-step business state machines into JSON."
        )
    ],
})
print(result)


The major trap of moving logic to configuration is embedding multi-step business state machines into JSON.


In [41]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"Moving logic to configuration carries a major trap: never embed multi-step business state machines into JSON.",
    "context":[Document(page_content="Moving logic to configuration carries a major trap: never embed multi-step business state machines into JSON.")]
})

'The statement provided is correct based on the context: moving logic to configuration carries a major trap, which is to never embed multi-step business state machines into JSON.'

In [43]:
retriever=vectorstoredb.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)

In [44]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001FF673BAF90>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='Answer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\nQuestion: {input

In [46]:
## Get the response from the llm
response = retrieval_chain.invoke({
    "input": "Moving logic to configuration carries a major trap: never embed multi-step business state machines into JSON."
})
response['answer']

'According to the provided context, the major trap of moving logic to configuration is that it can lead to building an unmaintainable programming language inside YAML. Specifically, you should never embed multi-step business state machines into JSON because mechanics such as control flow, interface definitions, and error handling belong in strictly typed, unit-tested application code, while only variables like thresholds, prompt IDs, model aliases, and weights should be kept in the data/config layer.'

In [48]:
response


{'input': 'Moving logic to configuration carries a major trap: never embed multi-step business state machines into JSON.',
 'context': [Document(id='88792995-2513-44a5-b635-32636dd2149b', metadata={'source': 'https://medium.com/@musabzafar63/building-large-ai-systems-is-an-architecture-problem-not-a-model-problem-42ab622e712f', 'title': 'Medium', 'description': 'Building Large AI Systems Is an Architecture Problem Not a Model Problem Why your clean 50-line GenAI prototype turned into a 3,000 line god class and the five architectural pillars required to fix …', 'language': 'en'}, page_content='Config Delusion: Moving logic to configuration is intoxicating until you’ve inadvertently built an unmaintainable programming language inside YAML. If a rule requires conditional branching, nested state, or mathematical evaluation, it belongs in a typed language with unit tests.Optimizing for Generation Over Retrieval: Teams routinely spend 80% of their time fine-tuning generation prompts and 20% 

In [49]:
response['context']

[Document(id='88792995-2513-44a5-b635-32636dd2149b', metadata={'source': 'https://medium.com/@musabzafar63/building-large-ai-systems-is-an-architecture-problem-not-a-model-problem-42ab622e712f', 'title': 'Medium', 'description': 'Building Large AI Systems Is an Architecture Problem Not a Model Problem Why your clean 50-line GenAI prototype turned into a 3,000 line god class and the five architectural pillars required to fix …', 'language': 'en'}, page_content='Config Delusion: Moving logic to configuration is intoxicating until you’ve inadvertently built an unmaintainable programming language inside YAML. If a rule requires conditional branching, nested state, or mathematical evaluation, it belongs in a typed language with unit tests.Optimizing for Generation Over Retrieval: Teams routinely spend 80% of their time fine-tuning generation prompts and 20% on ingestion. In production RAG, the majority of model errors stem from poor document extraction,'),
 Document(id='38260023-2b7a-4630-9